<h1 align="center">Machine Learning Workflow Automation</h1>

## Table of Contents
1. [Overview](#Overview)
2. [Setting up](#Setting-up)
3. [AWS Lambda](#AWS-Lambda)
   * [Introduction](#Introduction)
   * [Lambda functions](#Lambda-functions)
   * [Invoking a Lambda function](#Invoking-a-Lambda-function)
4. [AWS Step Functions](#AWS-Step-Functions)
   * [Introduction](#Introduction)
   * [JSONPath](#JSONPath)
   * [Setting up](#Setting-up)
   * [Creating a workflow](#Creating-a-workflow)
   * [Invoking a workflow](#Invoking-a-workflow)
5. [Epilogue](#Epilogue)

## Overview

A machine learning (ML) workflow is the structured, end-to-end process of developing and deploying an ML model. The key components of any ML workflow are data collection and pre-processing, model selection, model training and testing, and deployment. It is usually event-driven, meaning the end of one activity provokes the start of the next. 
<center><img src="img/ml-workflow.png" width="80%"></center>

As you have practiced the previous tutorial, you are now familiar with executing different components of a machine learning workflow in Amazon SageMaker AI. However, they are done manually and seperately. Additionally, you often need to repeat them multiple times to fix errors, improve your model and achieve desired performance. This process is error-prone and time-consuming. 

In this tutorial, you will learn how to automate these components and chain them together, ultimately forming a seamless ML workflow. By the end of this tutorial, you will be able to:
* Create an ML workflow with AWS Step Functions.
* Automate ML workflows with AWS Lambda.

## Setting up

Earlier, you created an execution role for this notebook instance, which provided access to Amazon SageMaker AI and Amazon S3. For the next sections, you need to extend this role's permissions to include access to additional services such as AWS Lambda, AWS Step Functions, and AWS IAM. To do this:
1. Head to IAM using the search bar.
<center><img src="img/iam.png" width="60%"></center>

2. In the left navigation sidebar, go to *Access Management* → *Roles*. Then, click on the name of the role this notebook instance is currently using. <center><img src="img/execution-role.png" width="80%"></center>
    If you forget the name of the role, simply run the code below to retrieve it.

In [3]:
from sagemaker import get_execution_role

get_execution_role().rsplit('/', 1)[-1]

'AmazonSageMaker-ExecutionRole-20250515T014982'

3. From the role details console, go to *Add permissions* → *Attach policies*.
4. Search for and select *IAMFullAccess*, *AWSLambda_FullAccess*, *AmazonS3FullAccess* and *AWSStepFunctionsFullAccess*. Scroll down and click on *Add permissions* button.
<center><img src="img/permissions.png" width="80%"></center>

## AWS Lambda

### Introduction

AWS Lambda is a serverless compute service that runs a single, self-contained function in response to events. It allows you to execute code without specifying underlying infrastructure, like hardware specifications, the operating system, or the maintenance of standard libraries. This service is ideal for small tasks that are frequently repeated.

In the programming realm, it is common to encounter situations where code that works on one machine often fails on another. Even in the managed services like Amazon SageMaker AI, you can still run into failures when running under different configurations. This is not the case in AWS Lambda.

### Lambda functions

A Lambda function is a piece of code that runs in response to events. When invoked, Lambda runs the function handler to process events and return necessary response. The handler function is the entry point of a Lambda function and other AWS services will use it to interact with your code. While the code in a Lambda function can  ontain more than one function, it can only have one function handler.

The handler function must match the name and structure expected by the runtime environment. For Python, the function must have:
* The exact name as configured. Conventionally, it is `lambda_handler`.
* First parameter: `event`, a dictionary containing the event data that triggered the Lambda function.
* Second parameter: `context`, an object provided by AWS Lambda that gives [methods and properties](https://docs.aws.amazon.com/lambda/latest/dg/python-context.html) to access information about the invocation, function, and execution environment.
* Optional response to return, in the form of a dictionary containing status message.

```python
def lambda_handler(event, context):
    # Your code here
    return {
        'statusCode': 200,
        'body': 'Insert your message'
    }
```

In this exercise, you're going to create a basic Lambda function in Python. Just like your notebook instace, your Lambda function requires an execution role of its own to access necessary AWS services. You can create this role using the IAM console, as you did earlier, or you can use `boto3`. For a complete list of available methods and detailed documentation on how to interact with IAM using `boto3`, visit [here](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/iam.html).

In [1]:
import boto3
import json

iam = boto3.client('iam')
# Construct trust policy to allow Lambda to use the role
trusted_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {
            "Service": "lambda.amazonaws.com"
        },
        "Action": "sts:AssumeRole"
    }]
}
# ARN of permission policies to let your Lambda function access desired AWS services
policies = ['arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
            'arn:aws:iam::aws:policy/AmazonS3FullAccess']
role_name = 'LambdaPreprocess'  # Enter a unique name for the role

iam_response = iam.create_role(RoleName=role_name,
                               AssumeRolePolicyDocument=json.dumps(trusted_policy))
for policy in policies:
    iam.attach_role_policy(RoleName=role_name, PolicyArn=policy)
print('ARN of the role:', iam_response['Role']['Arn'])

Next, your Lambda function will need a deployment package that contains your function code. Lambda supports two types of deployment packages: container images and zip file archives. In this exercise, you will package your Python script into a zip file and upload its binary contents directly to Lambda. The function below will help you achieve such task.

In [2]:
from zipfile import ZipFile


def get_deployment_package(source_code,
                           script_filename='lambda_function.py',
                           zip_filename='test_code.zip'):
    '''Package your source code into a deployment package (.zip file archives)
       for a Lambda function.'''

    with open(script_filename, 'w') as f:
        f.write(source_code)
    with ZipFile(zip_filename, 'w') as f:
        f.write(script_filename)
    # Get raw (binary) data of the zip file
    with open(zip_filename, 'rb') as f:
        raw_zip_file = f.read()
    return raw_zip_file

To create and interact with a Lambda function programmatically, you will need to instantiate a low-level Lambda client with `boto3`. For a complete list of available methods and detailed documentation on how to interact with Lambda, visit [here](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/lambda.html).

Since the deployment package is a zip file archive, you need to define handler setting. For Lambda functions using Python runtime, it's typically in the format `filename.function_name`:
* `filename` must be the name of your Python script (without the `.py` extension).
* `function_name` must be the name of the handler function defined within that script.

By convention, this is usually set to `lambda_function.lambda_handler`.

For starter, a dummy function code has been made for you. It only contains a function handler named `lambda_handler` and will be written in a Python file named `lambda_function.py`. Your task is to configure your Lambda function with approriate settings, such as function name, ARN of the role for the function, handler format, timeout in seconds (3 minutes is advisable for upcoming experiments).

In [3]:
aws_lambda = boto3.client('lambda')
# Enter your Lambda function's name
function_name = 'blazingtext-data-preprocess'
code = '''
def lambda_handler(event, context):
    return {
        'statusCode': 200,
        'body': 'Hello from Lambda!'
    }
'''
zip_file = get_deployment_package(code)

lambda_response = aws_lambda.create_function(FunctionName=function_name,
                                             Runtime='python3.13',
                                             Role=iam_response['Role']['Arn'],
                                             Handler='?',
                                             PackageType='Zip',
                                             Code={'ZipFile': zip_file},
                                             Timeout=?)
print('ARN of the Lambda function:', lambda_response['FunctionArn'])

Now, you will start testing your Lambda function. An empty payload will be sent to the function and it will respond back with a payload. However, this payload needs to be decoded in order to see the response message inside.

In [4]:
def decode_payload(response):
    return json.loads(response['Payload'].read().decode())


response = aws_lambda.invoke(FunctionName=function_name, Payload='{}')
decode_payload(response)

{'errorMessage': "'Records'",
 'errorType': 'KeyError',
 'requestId': '6b0043de-6ae7-44fd-949b-1373c0ae316b',
 'stackTrace': ['  File "/var/task/lambda_function.py", line 92, in lambda_handler\n    for record in event[\'Records\']:\n']}

You will soon receive the expected response message from the function. After that, replace its code with the new one written below. This code has a technical error that will cause your Lambda function to fail.

In [9]:
code = '''
def lambda_handler(event, context):
    try:
        assert(True == False)
    except:
        raise
    return {
        'statusCode': 200,
        'body': 'Hello from Lambda!'
    }
'''
aws_lambda.update_function_code(FunctionName=function_name,
                                ZipFile=get_deployment_package(code))
response = aws_lambda.invoke(FunctionName=function_name, Payload='{}')
decode_payload(response)

{'errorMessage': '',
 'errorType': 'AssertionError',
 'requestId': '3ace1642-57a1-4009-895d-589fd4ec5598',
 'stackTrace': ['  File "/var/task/lambda_function.py", line 4, in lambda_handler\n    assert(True == False)\n']}

From the decoded payload above, you can see the error message from the function. However, if an error occurred outside Lambda environment, e.g. algorithmic error from a processing job, and you want to see what the issue is, view the CloudWatch Logs regarding your Lambda function.
<center><img src="img/error-log.png" width="80%"></center>

### Invoking a Lambda function

There are two types of Lambda function invocations:
* Synchronous invocation will launch the function and expect a immediate response from the function. As a general rule, you'll be invoking Lambda functions synchronously in ML engineering operations.
<center><img src="img/invocation-sync.png" width="40%"></center>

* Asynchronous invocation does not wait for a response from the function, and when you submit an event, you are actually submitting it in a queue to be processed. This type of invocations is used when you don't want to have to handle all of the replies, and you may not even need to handle it.
<center><img src="img/invocation-async.png" width="60%"></center>

It is important to keep these two ways of invoking a Lambda function in mind and to consult the documentation to confirm which of these options are available to you.

As seen earlier, a Lambda function is triggered by an invocation event. The event sends a JSON object, but its structure differs depending on the service sending it. When configuring a service to send events to a Lambda function, consult the AWS documentation to determine the structure of the events your function will process.

Before trying these two invocations, you need to change your Lamda function's code into a data preprocessing script, similar to the one used in your processing jobs. Alternatively, you can use [code.zip](./code.zip) provided to override your Lambda function's code.\
If you decide to implement your own script, remember to properly define the handler function, as your Lambda function will only execute this function. The name of the handler function and the name of the Python file containing it must match what has been configured in the handler setting earlier. If needed, you can include multiple helper functions in your code. Moreover, your code should directly download the raw dataset using its S3 location extracted from the event argument, which will be explained later, and should directly upload training set and testing set into your S3 bucket. These actions requires you to work with `boto3` module. Lastly, Lambda function restricts your file operations within the local directory `/tmp/`. Hence, all files created, downloaded, or extracted must be in this folder.

Modify and run the code below to upload the dataset [`reviews_Musical_Instruments_5.json.zip`](../data/reviews_Musical_Instruments_5.json.zip) to your S3 bucket for the next experiments.

In [5]:
s3 = boto3.client('s3')
# Enter your bucket name
bucket_name = 'ml-workflow-2'
# Key name for the dataset, not S3 path as explained in the first tutorial
key_name = 'data/reviews_Musical_Instruments_5.json.zip'
s3.upload_file('../data/reviews_Musical_Instruments_5.json.zip', bucket_name, key_name);

The JSON object below will simulate an event data that is sent to a Lambda function when a file is uploaded into an observed S3 bucket. This is what will be sent to your Lambda function’s handler, via the `event` argument. Your function handler should be able to extract the values inside the *name* field under the *bucket* field and the *key* field under the *object* field. \
Note that the *Records* field in the event payload contains a list, not subfields. This allows AWS S3 to notify a Lambda function about multiple object-related events in a single invocation. In this case, it would be two or more files uploaded simultaneously. Therefore, your code should iterate over this list to handle each individual record.

In [12]:
event_json = {
 'Records': [{'eventVersion': '2.0',
              'eventSource': 'aws:s3',
              'awsRegion': 'us-east-1',
              'eventTime': '1970-01-01T00:00:00.000Z',
              'eventName': 'ObjectCreated:Put',
              'userIdentity': {'principalId': 'EXAMPLE'},
              'requestParameters': {'sourceIPAddress': '127.0.0.1'},
              'responseElements': {'x-amz-request-id': 'EXAMPLE123456789',
                                   'x-amz-id-2': 'EXAMPLE123/5678abcdefghijklambdaisawesome/mnopqrstuvwxyzABCDEFGH'},
              's3': {'s3SchemaVersion': '1.0',
                     'configurationId': 'testConfigRule',
                     'bucket': {'name': bucket_name,  # Name of your bucket
                                'ownerIdentity': {'principalId': 'EXAMPLE'},
                                'arn': 'arn:aws:s3:::example-bucket'},
                     'object': {'key': key_name,  # Key name of your newly uploaded dataset
                                'size': 1024,
                                'eTag': '0123456789abcdef0123456789abcdef',
                                'sequencer': '0A1B2C3D4E5F678901'}
                     }
              }]
}

After learning about the event JSON, it's time to update the code of your Lambda function. Put your data preprocessing script into a zip file and upload its binary contents directly to your function with the code below.

In [13]:
# Enter name of the zip file containing your script
zip_filename = 'code.zip'
with open(zip_filename, 'rb') as f:
    raw_zip_file = f.read()

aws_lambda.update_function_code(FunctionName=function_name, ZipFile=raw_zip_file);

{'ResponseMetadata': {'RequestId': 'c39cb290-6770-430a-a009-481ce9fafd20',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 18 Jul 2025 23:10:10 GMT',
   'content-type': 'application/json',
   'content-length': '1378',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c39cb290-6770-430a-a009-481ce9fafd20'},
  'RetryAttempts': 0},
 'FunctionName': 'blazingtext-data-preprocess',
 'FunctionArn': 'arn:aws:lambda:us-east-1:988060366449:function:blazingtext-data-preprocess',
 'Runtime': 'python3.13',
 'Role': 'arn:aws:iam::988060366449:role/LambdaPreprocess',
 'Handler': 'lambda_function.lambda_handler',
 'CodeSize': 1298,
 'Description': '',
 'Timeout': 300,
 'MemorySize': 128,
 'LastModified': '2025-07-18T23:10:10.000+0000',
 'CodeSha256': '+gnIKeAn/k/HnbeqJGal+BIOPeP2w+MYnHrVtCTr1Hg=',
 'Version': '$LATEST',
 'TracingConfig': {'Mode': 'PassThrough'},
 'RevisionId': '759444aa-5b7a-4387-b9a2-209c9ef2db83',
 'State': 'Active',
 'LastUpdateStatus': 'InProgress',
 'LastUpdateStatu

To trigger your Lambda function synchronously, you can use the `invoke()` method, just as you did previously. After a few minutes, you will receive the response. If it fails, try to fix the problem showed in the response message and in the CloudWatch Logs. If it succeeds, you will be able to see training set and testing set of the new dataset in your S3 bucket.

In [34]:
# Function to list all key names of a bucket
def list_key_names(bucket_name):
    bucket = boto3.resource('s3').Bucket(bucket_name)
    for obj in bucket.objects.all():
        print(obj.key)


response = aws_lambda.invoke(FunctionName=function_name, Payload=json.dumps(event_json))
print(decode_payload(response))
list_key_names(bucket_name)

{'statusCode': 200, 'body': 'PREPROCESS SUCCESSFULLY!'}
data/
data/custom_reviews.json
data/hello_blaze_preprocess.py
data/reviews_Musical_Instruments_5.json.zip
data/reviews_Toys_and_Games_5.json.zip
data/test/reviews_Musical_Instruments_5_test.txt
data/train/reviews_Musical_Instruments_5_train.txt
models/


Next, you will trigger this Lambda function asynchronously with real events. First, grant S3 service the permission to invoke your Lambda function. Start by granting the S3 service permission to invoke your Lambda function. Then, add an event notification to your bucket so that it invokes the function only when objects are uploaded. Additionally, include filters in the configuration to restrict invocation to objects with a specific prefix and file type (suffix).

Please be aware of the risk of recursive invocations. They happen when your Lambda function is automatically triggered by its own uploads. Specifically, when uploaded files by your function reside in the same bucket and match the prefix as well as the file type you specified in filter setting. 

In [10]:
# Grant S3 permission to invoke your Lambda function
aws_lambda.add_permission(FunctionName=function_name,
                          StatementId='S3Trigger',  # Unique identifier
                          Principal='s3.amazonaws.com',  # Entity that can access the function
                          Action='lambda:InvokeFunction',  # Action to be use on the function
                          SourceArn=f'arn:aws:s3:::{bucket_name}'  # Set your bucket as event source
                          )

# Set the S3 event notification configuration
notification_config = {
    'LambdaFunctionConfigurations': [{
        'LambdaFunctionArn': ?,
        # Trigger Lambda function only when new objects are uploaded
        'Events': ['s3:ObjectCreated:Put',
                   's3:ObjectCreated:CompleteMultipartUpload'],
        'Filter': {
            'Key': {
                'FilterRules': [
                    {'Name': 'prefix', 'Value': ?},
                    {'Name': 'suffix', 'Value': '.json.zip'}
                ]
            }
        }
    }]
}

s3.put_bucket_notification_configuration(Bucket=bucket_name,
                                         NotificationConfiguration=notification_config);

{'ResponseMetadata': {'RequestId': 'DQBJB18M124P8GQD',
  'HostId': '1oGpvDGdU67KdCVAFIGtcNwdnXkZwwXekZkmAss+V4q5Vfe00A16IdnNxsZ+KQ+rJ3E0DFuZJGVlWSOxvDHnUg==',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '1oGpvDGdU67KdCVAFIGtcNwdnXkZwwXekZkmAss+V4q5Vfe00A16IdnNxsZ+KQ+rJ3E0DFuZJGVlWSOxvDHnUg==',
   'x-amz-request-id': 'DQBJB18M124P8GQD',
   'date': 'Sun, 20 Jul 2025 00:51:17 GMT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

Upon uploading the dataset [`reviews_Patio_Lawn_and_Garden_5.json.zip`](../data/reviews_Patio_Lawn_and_Garden_5.json.zip) in the directory path (prefix) you specified earlier, your Lambda function will be triggered. After a moment, you will be able to see the training set and the testing set of the new dataset in your bucket.

In [35]:
key_name = 'data/reviews_Patio_Lawn_and_Garden_5.json.zip'
s3.upload_file('../data/reviews_Patio_Lawn_and_Garden_5.json.zip', bucket_name, key_name)

In [37]:
# Wait for about 2 minutes before running this code
list_key_names(bucket_name)

data/
data/custom_reviews.json
data/hello_blaze_preprocess.py
data/reviews_Musical_Instruments_5.json.zip
data/reviews_Patio_Lawn_and_Garden_5.json.zip
data/reviews_Toys_and_Games_5.json.zip
data/test/reviews_Musical_Instruments_5_test.txt
data/test/reviews_Patio_Lawn_and_Garden_5_test.txt
data/train/reviews_Musical_Instruments_5_train.txt
data/train/reviews_Patio_Lawn_and_Garden_5_train.txt
models/


Unlike synchronous invocation, you cannot receive a direct response message from your Lambda function with asynchronous invocation. However, you are still able to inspect error messages in CloudWatch Logs.

<ins>**Please note**</ins>: You are billed for every Lambda function invocation, even if it is a test event. Therefore, refrain yourself from spamming invocations and delete all triggers after you finish your experiment.

In [ ]:
aws_lambda.remove_permission(FunctionName=function_name,
                             StatementId='S3Trigger')
# Empty all existing event notifications
s3.put_bucket_notification_configuration(Bucket=bucket_name,
                                         NotificationConfiguration={});

Similarly, you can construct a Lambda function to automate a workflow in AWS Step Functions with invocations.

## AWS Step Functions

### Introduction

AWS Step Functions is a visual workflow service that helps you use AWS services to build distributed applications, automate processes, orchestrate microservices, and create ML workflows.

It is based on two abstractions:
* State machine: serverless workflow, which is a series of event-driven steps.
* Task: a state within a workflow that represents a single unit of work performed by another AWS service.

<center><img src="img/concept.png" width="60%"></center>

Advantages:
* Intuitive UI.
* Easy visualization.
* Easy isolation of failure points.

Disadvantages:
* Significantly more expensive than Lambda function.
* Dependent on proprietary Amazon State Language.
* Not compatible with similar orchestration tools in other platforms.

### JSONPath

From this point forward, you are going to use `stepfunctions` module, which is a higher-level AWS SDK specifically designed for AWS Step Functions. As usual, you are encouraged to explore [its documentation](https://aws-step-functions-data-science-sdk.readthedocs.io/en/stable/index.html) for future practice.

The Amazon States Language (ASL) is a JSON-based, structured language used to define your state machine. It mostly resembles a nested dictrionary in Python, from the syntax to the structure, with the exception of query langague embedded. 

ASL originally employs JSONPath as its query language. However, it recently adopts [JSONata](https://docs.jsonata.org/overview.html) as the default query language. When a state machine execution receives JSON input from execution, it passes that data to the first state in the workflow as input. With JSONata, you can retrieve a state input from `states.input`. If a state is successfully completed, its API response will be stored in `states.result`. Furthermore, you can use JSONata to dynamically configure your state machine during each execution.
<center><img src="img/vars-jsonata.png" width="80%"></center>

In this exercise, you will embed JSONata expression inside a string value. The expression must start with `{%` with no leading spaces, and must end with `%}` with no trailing spaces. Improperly opening or closing the expression will result in a validation error.
```JSON
"{% <JSONata expression> %}"
```
Any JSONata functions, provided by Step Functions, or variables must start with the character `$`, e.g. `$states.result` or `$split($variable_1)[0]`. Moreover, because the JSONata expression is in a string value, to add a string into the expression, use the single quote `'` or `\"`. Finally, to concatenate string, use the ampersand character `&`. For example:
```JSON
"{% $states.input.prefix & 'train/' & $split($states.input.dataset, '.', 1)[0] & '_train.txt' %}"
```

No worries, this tutorial will help you set up an ML workflow with ASL along the way. For more information, read [this](https://docs.aws.amazon.com/step-functions/latest/dg/transforming-data.html#writing-jsonata-expressions-in-json-strings) and watch [this](https://www.youtube.com/watch?v=kVWxJoO_zc8).


### Creating a workflow

First of all, you need to create a role and your state machine will use it to gain full access to SageMaker AI and CloudWatch Events. Your state machine needs access to Amazon CloudWatch Events (now part of Amazon EventBridge) so that it can send logs and events to CloudWatch. This enables you to monitor, debug, and manage your workflow.

In [2]:
# Trust policy to allow Step Functions to use the role
trusted_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {
            "Service": "states.amazonaws.com"
        },
        "Action": "sts:AssumeRole"
    }]
}
# ARN of permission policies to let your state machine access desired AWS services
policies = ['arn:aws:iam::aws:policy/AmazonSageMakerFullAccess',
            'arn:aws:iam::aws:policy/CloudWatchEventsFullAccess']
role_name = 'MLWorkflow-statemachine'  # Enter a unique name for the role

iam_response = iam.create_role(RoleName=role_name,
                               AssumeRolePolicyDocument=json.dumps(trusted_policy))
for policy in policies:
    iam.attach_role_policy(RoleName=role_name, PolicyArn=policy)
print('ARN of the role:', iam_response['Role']['Arn'])

ARN of the role: arn:aws:iam::988060366449:role/MLWorkflow-statemachine


From this point forward, you are going to use `stepfunctions` module, which is a higher-level AWS SDK specifically designed for AWS Step Functions. Finally, you are encouraged to explore [its documentation](https://aws-step-functions-data-science-sdk.readthedocs.io/en/stable/index.html) for future practice.

In [4]:
%pip install stepfunctions

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'stepfunctions' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'stepfunctions'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for stepfunctions: filename=stepfunctions-2.3.0-py2.py3-none-any.whl size=78237 sha256=d29296d8fb0f4c748ad6f505a356348c04db974131d4ee6df9845f625c0a0773
  Stored in directory: /home/ec2-user/.cache/pip/wheels/e8/8e/d2/b36300e3b204b743131942831a469a45ee5e5180884f7306c5
Successfully built stepfunctions
Note: you may need to restart the kernel to use updated packages.


In [9]:
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor

# Get the execution role
role = sagemaker.get_execution_role()
# Get current session
session = sagemaker.Session()

# Create an SKLearnProcessor, version 1.2-1
sklearn_processor = SKLearnProcessor(framework_version='1.2-1',
                                     role=role,
                                     instance_type='ml.m5.large',
                                     instance_count=1,
                                     sagemaker_session=session)

container_prefix = '/opt/ml/processing/'
# Local directory path that the dataset will be downloaded into
input_path = container_prefix + 'input/data'
# Local directory path that the Python script will be downloaded into
code_path = container_prefix + 'input/code'
# Local directory paths where your Python script saves the training/testing set
train_path = container_prefix + 'output/train'
test_path = container_prefix + 'output/test'

print("Raw dataset will be downloaded to", input_path)
print("Python script will be downloaded to", code_path)
print("Python script saves training set in", train_path)
print("Python script saves testing set in", test_path)

Raw dataset will be downloaded to /opt/ml/processing/input/data
Python script will be downloaded to /opt/ml/processing/input/code
Python script saves training set in /opt/ml/processing/output/train
Python script saves testing set in /opt/ml/processing/output/test


To make it simple, easy, and dynamic, enter the leading string of your choice for the name and concatenate it with

In [7]:
params = {
  "ProcessingJobName": "{% 'BlazingText-' & $millis() %}",
  "RoleArn": "arn:aws:iam::988060366449:role/service-role/AmazonSageMaker-ExecutionRole-20250515T014982",
  "ProcessingResources": {
    "ClusterConfig": {
      "InstanceCount": 1,
      "InstanceType": "ml.m5.large",
      "VolumeSizeInGB": 1
    }
  },
  "ProcessingInputs": [
    {
      "InputName": "input",
      "S3Input": {
        "S3DataType": "S3Prefix",
        "S3InputMode": "File",
        "S3Uri": "{% $states.input.prefix & $states.input.dataset %}",
        "LocalPath": "/opt/ml/processing/input/data"
      }
    },
    {
      "InputName": "code",
      "S3Input": {
        "S3DataType": "S3Prefix",
        "S3InputMode": "File",
        "S3Uri": "s3://ml-workflow-1/data/hello_blaze_preprocess.py",
        "LocalPath": "/opt/ml/processing/input/code"
      }
    }
  ],
  "AppSpecification": {
    "ImageUri": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
    "ContainerEntrypoint": [
      "python3",
      "/opt/ml/processing/input/code/hello_blaze_preprocess.py"
    ],
    "ContainerArguments": [
      "{% $states.input.dataset %}"
    ]
  },
  "ProcessingOutputConfig": {
    "Outputs": [
      {
        "OutputName": "train_set",
        "S3Output": {
          "S3Uri": "{% $states.input.prefix & 'train/' %}",
          "LocalPath": "/opt/ml/processing/output/train",
          "S3UploadMode": "EndOfJob"
        }
      },
      {
        "OutputName": "test_set",
        "S3Output": {
          "S3Uri": "{% $states.input.prefix & 'test/' %}",
          "LocalPath": "/opt/ml/processing/output/test",
          "S3UploadMode": "EndOfJob"
        }
      }
    ]
  }
}

In [19]:
from sagemaker.processing import ProcessingInput, ProcessingOutput
from stepfunctions.inputs import ExecutionInput
from stepfunctions.steps.sagemaker import ProcessingStep
from stepfunctions.steps.states import Pass, Chain

execution_input = ExecutionInput(schema={
    "bucket": str,  # "your-bucket"
    "prefix": str,  # "s3://your-bucket/data/"
    "dataset": str,  # "dataset.json.zip"
    "timestamps": str
})

# Construct S3 location of your dataset from the execution input
dataset_s3 = "{% $states.input.prefix & $states.input.dataset %}"
# Enter S3 location of your Python script
code_s3 = "s3://ml-workflow-2/data/hello_blaze_preprocess.py"
# S3 path where you want SageMaker AI to upload the training/testing set
train_s3 = "{% $states.input.prefix & 'train/' %}"
test_s3 = "{% $states.input.prefix & 'test/' %}"

inputs = [ProcessingInput(source=dataset_s3, destination=input_path, input_name ='dataset'),
          ProcessingInput(source=code_s3, destination=code_path, input_name ='code')]
outputs = [ProcessingOutput(source=train_path, destination=train_s3, output_name='train'),
           ProcessingOutput(source=test_path, destination=test_s3, output_name='validation')]

processing_step = ProcessingStep("SageMaker Pre-processing",
                                 processor=sklearn_processor,
                                 job_name="{% 'BlazingText-' & $millis() %}",
                                 inputs=inputs,
                                 outputs=outputs,
                                 container_entrypoint=["python3", code_path+"/"+code_s3.split('/')[-1]],
                                 )
pass_state_with_jsonata = Pass(
    state_id="TransformDataWithJSONata",
    parameters={
        "QueryLanguage": "JSONata"
    }
)


In [21]:
from stepfunctions.workflow import Workflow

workflow = Workflow(
    name="hello-blaze-workflow",
    definition=Chain([pass_state_with_jsonata, processing_step]),
    role="arn:aws:iam::988060366449:role/MLWorkflow-statemachine")

workflow.create()


'arn:aws:states:us-east-1:988060366449:stateMachine:hello-blaze-workflow'

In [22]:
workflow.delete()

## Epilogue

Congratulations! You have successfully built and automated your first machine learning workflow. This marks a significant milestone in building production-ready, event-driven ML solutions using AWS services. 

As a best practice, __remember to delete or turn off__ any cost-consuming instances in Amazon SageMaker AI and AWS Lambda. Moreover, you are encouraged to practice what you have learned so far to a different use case.